# Lesson 27: Instruction Fine-Tuning Microsoft Phi-3.5-mini-instruct

This notebook demonstrates how to fine-tune **Microsoft Phi-3.5-mini-instruct** on a custom instruction dataset using Hugging Face `transformers`, `datasets`, and the `Trainer` API.

### Key Concepts:
1. **Hugging Face Chat Templates**: Using `tokenizer.apply_chat_template()` for consistent instruction/dialogue formatting.
2. **Prompt Loss Masking**: Setting user prompt token labels to `-100` so that cross-entropy loss is computed strictly on assistant generation tokens.
3. **Dynamic Batch Padding**: Using `DataCollatorForSeq2Seq` to pad batches on the fly while properly ignoring padded labels.
4. **Memory Efficient Training**: Utilizing FP16 precision, gradient accumulation, and learning rate scheduling.

## 1. Imports & Environment Setup

In [ ]:
import json
import os
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)

## 2. Configuration & Hyperparameters

In [ ]:
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
MAX_LENGTH = 512
OUTPUT_DIR = "./phi-3.5-mini-instruct"

# Fallback path logic to support running from any directory
DATA_PATH = "/Users/mac/Desktop/Machine Learning/DL/Homework/instruction-data.json"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "instruction-data.json"

## 3. Load Instruction Dataset

In [ ]:
print("Loading instruction dataset...")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
    
print("Number of examples:", len(data))
print("First example:")
print(json.dumps(data[0], indent=2))

## 4. Load Model & Tokenizer
We load Phi-3.5-mini-instruct in `float16` precision with `trust_remote_code=True` and automated device mapping (`cuda`, `mps`, or `cpu`).

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Tokenizer vocabulary size:", len(tokenizer))

print("\nLoading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully.")
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Model device:", model.device)

## 5. Format Conversations into Chat Template Schema
We map each JSON instruction example into standard conversational message blocks with `'user'` and `'assistant'` roles.

In [ ]:
def format_conversation(example):
    instruction = example["instruction"].strip()
    input_text = example.get("input", "").strip()
    output = example["output"].strip()
    
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction
        
    messages = [
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": output
        }
    ]
    
    return {"messages": messages}

# Create Hugging Face Dataset
dataset = Dataset.from_list(
    [format_conversation(example) for example in data]
)

print("Sample formatted conversation:")
print(dataset[0]["messages"])

## 6. Tokenization & Prompt Loss Masking
We use `tokenizer.apply_chat_template()` to format the full dialogue and the prompt prefix separately. All tokens belonging to the user prompt are replaced with `-100` in the target labels array, ensuring the model only learns to predict the assistant's response.

In [ ]:
def prepare_input(example):
    messages = example["messages"]
    
    # Apply chat template to full dialogue
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    
    # Apply chat template to prompt prefix only
    user_messages = [messages[0]]
    prompt_text = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize sequences
    full_encoding = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    prompt_encoding = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    input_ids = full_encoding["input_ids"]
    attention_mask = full_encoding["attention_mask"]
    
    prompt_length = min(
        len(prompt_encoding["input_ids"]),
        len(input_ids)
    )
    
    # Mask user prompt tokens with -100
    labels = input_ids.copy()
    for i in range(prompt_length):
        labels[i] = -100
        
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("Tokenizing dataset...")
tokenized_ds = dataset.map(
    prepare_input,
    remove_columns=["messages"]
)

print("Tokenization completed.")
print("Number of examples:", len(tokenized_ds))

# Check label masking on first sample
first_example = tokenized_ds[0]
print("\nFirst example verification:")
print("Input IDs (first 30):", first_example["input_ids"][:30])
print("Labels (first 30):   ", first_example["labels"][:30])
print(
    "Tokens active for loss calculation:",
    sum(label != -100 for label in first_example["labels"])
)

## 7. Data Collator & Training Configuration
We use `DataCollatorForSeq2Seq` which handles dynamic padding and sets padded labels to `-100`.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.05, # type: ignore
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
    processing_class=tokenizer
)

## 8. Fine-Tuning Execution

In [ ]:
print("Starting instruction fine-tuning...")
trainer.train()

## 9. Save Fine-Tuned Model & Tokenizer

In [ ]:
print("Saving fine-tuned model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training completed.")
print("Model saved to:", OUTPUT_DIR)